# 🎵 ARIS: 从零到一完整教程 (From Scratch to Acoustic Manipulation)
### 神经网络源–滤波器可微声码器：语音学实验刺激生成全流程

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/N1r/ARIS_nsf/blob/main/notebooks/ARIS_Tutorial_and_Workflow.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-ARIS__nsf-blue.svg)](https://github.com/N1r/ARIS_nsf)
[![Interactive Demo](https://img.shields.io/badge/Web-Listening_Demo-green.svg)](https://n1r.github.io/ARIS_nsf/)

---

### 📖 教程导读
本教程展示如何使用 ARIS 从零构建高保真语音分析合成管线，并生成严格受控的成对感知实验刺激：
- **标准 Python 交互**：通过直观的 Python 函数完成音频加载、基频 $F_0$ 轮廓提取、波形/语谱图绘制与声学模型训练。
- **声学维度正交解耦**：在保持其他声学参数严格不变的前提下，独立操控第一共振峰 $F_1$、基频音高 $F_0$ 与声门波形参数 $R_d$（气声与紧嗓）。
- **即时听觉与时频反馈**：每一步均内嵌音频播放器与时频对齐语谱图对比，并支持拉起网页端交互式工作台 (Web Studio)。

> **一键运行**：在 Colab 顶部菜单栏点击 **“代码执行程序” (Runtime) ➔ “全部运行” (Run all)**，即可按顺序完成全流程演示。

---

### 📋 工作流概览：
$$\\text{1. 环境安装} \\longrightarrow \\text{2. 下载音频与单样本探索} \\longrightarrow \\text{3. 提取F0与数据准备} \\longrightarrow \\text{4. 模型训练} \\longrightarrow \\text{5. 成果试听与语谱图} \\longrightarrow \\text{6. 语音学操控 (Manipulation)}$$


## 步骤 1. 🛠️ 环境安装 (Environment Setup)

Google Colab 提供了免费的 GPU 加速（T4/L4）。我们首先检查显卡，并拉取项目源码及全部依赖库。


In [ ]:
# 1.1 检查云端 GPU 状态
!nvidia-smi


In [ ]:
# 1.2 克隆仓库、配置 Python 模块路径并安装依赖
import os
import sys

# 若在 Google Colab 中运行，克隆主仓库并进入工作目录
if not os.path.exists("pyproject.toml"):
    !git clone https://github.com/N1r/ARIS_nsf.git
    %cd ARIS_nsf

# 将 src/ 加入模块导入路径，确保任何环境都能直接 import aris
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# 安装依赖项（音频、训练、Studio 等全部组件）
!pip install -q -e ".[all]"
print("✓ 项目路径就绪，依赖安装完成！")


In [ ]:
# 1.3 导入 aris 并运行内置诊断工具检查环境
import aris

print(f"ARIS 版本: {aris.__version__}\n")
for item in aris.doctor():
    mark = "✓ [OK]" if item.ok else "✗ [--]"
    print(f"{mark} {item.name:12} : {item.detail} ({item.required_for})")


---
## 步骤 2. 📥 下载音频数据与单样本探索 (Data & Exploration)

我们从 GitHub 官方 Release 下载配套示例包（包含普通话女声 F024 的 16 kHz 语音录音与训练好的模型检查点，约 65 MB）。
下载完成后，我们通过标准的 Python 代码**读取音频、提取基频 $F_0$ 轮廓并可视化**：


In [ ]:
# 2.1 下载并解压音频数据包 (ZIP)
import urllib.request
import zipfile
from pathlib import Path

data_zip = Path("aris_f024_demo.zip")
if not Path("demo_f024").exists():
    url = "https://github.com/N1r/ARIS_nsf/releases/download/v0.1.0/aris_f024_demo.zip"
    print(f"正在从 GitHub 下载示例数据包: {url} ...")
    urllib.request.urlretrieve(url, data_zip)
    print("解压中...")
    with zipfile.ZipFile(data_zip, "r") as zf:
        zf.extractall(".")
    print("✓ 数据与模型已解压到 demo_f024/")
else:
    print("✓ demo_f024/ 数据目录已就绪。")


In [ ]:
# 2.2 单音频探索：试听原声并用 aris.audio.estimate_f0 提取基频
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

from aris.audio import estimate_f0, read_audio

# 读取一个普通话语音样本
sample_wav = "demo_f024/dataset/audio/F024_ci1-29ba87b716.wav"
audio, sr = read_audio(sample_wav)

print(f"音频采样率: {sr} Hz | 采样点数: {len(audio)} | 时长: {len(audio)/sr:.2f} 秒")
print("🔊 试听原始音频录音：")
display(Audio(audio, rate=sr))

# 提取基频 F0 (使用 WORLD 算法)
f0, backend = estimate_f0(audio, sr, floor_hz=50, ceiling_hz=800, method="pyworld")
time_axis = np.linspace(0, len(audio) / sr, len(f0))

# 可视化：语音波形与基频 F0 曲线
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.plot(np.linspace(0, len(audio) / sr, len(audio)), audio, color="#2b5c8f")
ax1.set_title("语音波形 (Waveform)", fontsize=11, fontweight="bold")
ax1.set_ylabel("振幅")

# 仅在浊音段 (F0 > 0) 绘制音高曲线
f0_plot = np.where(f0 > 0, f0, np.nan)
ax2.plot(time_axis, f0_plot, color="#d95f02", linewidth=2.5)
ax2.set_title(f"基频轮廓 (F0 Track - {backend})", fontsize=11, fontweight="bold")
ax2.set_ylabel("频率 (Hz)")
ax2.set_xlabel("时间 (秒)")
ax2.set_ylim(50, 400)
plt.tight_layout()
plt.show()


---
## 步骤 3. ✂️ 数据集切分与准备 (Dataset Preparation)

在实际语音研究中，数据准备包含：
1. **音频重采样**至模型采样率（16 kHz）。
2. **批量提取 $F_0$ 基频轮廓**并保存为缓存。
3. **划分训练/验证/测试集**并生成 `manifest.csv`。

我们直接调用 Python 函数 `aris.prepare`：


In [ ]:
# 3.1 批量提取特征与划分数据集
manifest = aris.prepare(
    source="demo_f024/dataset/audio",
    output="data/my_prepared_dataset",
    sample_rate=16000,
    f0_method="pyworld",
    min_duration=0.5,
)

print(f"✓ 数据集准备完成，共索引 {len(manifest.records)} 条样本！")
# 查看前 3 条样本的元数据详情
for r in manifest.records[:3]:
    print(f"  ID: {r.id:25} | 分割: {r.split:10} | 时长: {r.duration_s:.2f}s | 中位F0: {r.median_f0_hz:.1f}Hz")


In [ ]:
# 3.2 校验数据集完整性
errors = aris.validate("data/my_prepared_dataset")
if not errors:
    print("✓ 数据集完整性校验全部通过！")
else:
    print("✗ 校验警告:", errors)


---
## 步骤 4. 🚀 模型配置与训练 (Model Training)

ARIS 采用 `aria-golf` 声码器模型（结合可微时变线性预测与声门波形合成），支持声源与滤波器的解耦控制。
我们调用 `aris.init_experiment` 和 `aris.train`：


In [ ]:
# 4.1 初始化实验配置
exp_dir = aris.init_experiment(
    dataset="data/my_prepared_dataset",
    output="experiments/colab_run",
    model="aria-golf",
    batch_size=16,
    max_steps=50,
)
print("✓ 实验配置文件已生成于:", exp_dir)


In [ ]:
# 4.2 启动快速训练循环演示（演示 GPU 迭代过程）
print("启动 GPU 训练循环...")
aris.train(exp_dir, extra_args=["--trainer.max_steps=50"])
print("✓ 训练演示完成！在真实研究中，通常训练 10,000~30,000 步（约 1~2 小时）即可达到最佳收敛。")


---
## 步骤 5. 📊 成果可视化与试听评估 (Speech Reconstruction)

现在我们调用 `aris.synthesize`，使用已充分收敛的预训练检查点对保留测试集语音进行**重建推理**，并通过波形与语谱图对比原声与合成声：


In [ ]:
# 5.1 使用预训练模型进行语音重建 (Inference)
checkpoint_path = "demo_f024/experiment/runs/checkpoints/last.ckpt"
recon_dir = Path("out/reconstruction")

aris.synthesize(
    experiment="demo_f024/experiment",
    checkpoint=checkpoint_path,
    output=recon_dir,
)
print(f"✓ 重建完成，WAV 文件保存在: {recon_dir}")


In [ ]:
# 5.2 原声 vs 重建语音 并排试听 (A/B 盲听)
test_stem = "F024_bian4"
orig_file = Path("demo_f024/dataset/audio") / f"{test_stem}-2e74da0c07.wav"
recon_file = recon_dir / f"{test_stem}-2e74da0c07.wav"

print("🔊 【原声】录音室原始发音：")
display(Audio(filename=str(orig_file)))

print("🔊 【ARIS 重建】神经网络声码器还原发音：")
display(Audio(filename=str(recon_file)))


In [ ]:
# 5.3 绘制时频对齐语谱图对比
import soundfile as sf

y_orig, sr_orig = sf.read(orig_file)
y_recon, sr_recon = sf.read(recon_file)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.specgram(y_orig, Fs=sr_orig, NFFT=512, noverlap=384, cmap="viridis")
ax1.set_title("【原始录音】语谱图", fontsize=11, fontweight="bold")
ax1.set_ylabel("频率 (Hz)")
ax1.set_ylim(0, 5000)

ax2.specgram(y_recon, Fs=sr_recon, NFFT=512, noverlap=384, cmap="viridis")
ax2.set_title("【ARIS 可微重建】语谱图", fontsize=11, fontweight="bold")
ax2.set_ylabel("频率 (Hz)")
ax2.set_xlabel("时间 (秒)")
ax2.set_ylim(0, 5000)

plt.tight_layout()
plt.show()


---
## 步骤 6. 🎛️ 语音学声学参数操控 (Manipulation)

这是语音学实验最核心的步骤：**在保持其他所有声学属性不变的前提下，独立改变特定声学特征**。

我们调用 `aris.manipulate`，生成 4 种经典的实验刺激变体：
- `f1_scale=1.2`：第一共振峰 $F_1$ 抬高 20%（模拟开口度变大 / 舌位降低）。
- `pitch_semitones=-4`：基频 $F_0$ 整体下调 4 个半音。
- `glottal_rd_scale=1.6`：声门波形参数增加（模拟气声/虚声 breathy voice）。
- `glottal_rd_scale=0.6`：声门波形参数减少（模拟紧喉/嘎裂声 creaky voice）。


In [ ]:
# 6.1 生成成对实验刺激
stimuli_dir = Path("out/stimuli")

aris.manipulate(
    experiment="demo_f024/experiment",
    checkpoint=checkpoint_path,
    output=stimuli_dir,
    variants=[
        "f1_up:f1_scale=1.2",
        "pitch_down:pitch_semitones=-4",
        "breathy:glottal_rd_scale=1.6",
        "creaky:glottal_rd_scale=0.6",
    ],
)
print("✓ 实验刺激生成完毕，保存在:", stimuli_dir)


In [ ]:
# 6.2 试听声学操控效果
f1_wav = stimuli_dir / "f1_up" / recon_file.name
pitch_wav = stimuli_dir / "pitch_down" / recon_file.name
breathy_wav = stimuli_dir / "breathy" / recon_file.name
creaky_wav = stimuli_dir / "creaky" / recon_file.name

print("🔊 1. 基准声音 (Baseline):")
display(Audio(filename=str(recon_file)))

print("🔊 2. 第一共振峰抬高 (+20% F1 Scale, 开口度变大):")
display(Audio(filename=str(f1_wav)))

print("🔊 3. 音高下移 (-4 Semitones):")
display(Audio(filename=str(pitch_wav)))

print("🔊 4. 气声音质 (Breathy Voice, Rd = 1.6):")
display(Audio(filename=str(breathy_wav)))

print("🔊 5. 紧喉/嘎裂音质 (Creaky Voice, Rd = 0.6):")
display(Audio(filename=str(creaky_wav)))


In [ ]:
# 6.3 语谱图对比：清晰查看 F1 共振峰抬高与基频下移
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for ax, wav, title in zip(
    axes,
    [recon_file, f1_wav, pitch_wav],
    ["基准声音 (Baseline)", "第一共振峰升高 (+20% F1 Scale)", "基频整体下移 (-4 Semitones)"]
):
    y, sr = sf.read(wav)
    ax.specgram(y, Fs=sr, NFFT=512, noverlap=384, cmap="magma")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel("频率 (Hz)")
    ax.set_ylim(0, 4500)

axes[-1].set_xlabel("时间 (秒)")
plt.tight_layout()
plt.show()


---
## 步骤 7. 🌐 网页交互式工作台 (Interactive Web Studio)

除了写代码调用，还可以直接在云端拉起 Gradio 交互式网页。它会生成一个公开链接（以 `gradio.live` 结尾），点击打开后即可在网页里用鼠标拖动滑块调音：


In [ ]:
# 启动可视化 Studio（点击输出的 public URL 即可在本地浏览器操作）：
# 运行时会持续提供服务，按单元格左侧的停止按钮即可退出。

aris.launch_studio(workspace=".", share=True, open_browser=False)


---
## 步骤 8. 📦 一键打包下载实验刺激 (Export Stimuli)

ARIS 为每次操控自动生成严格记录参数的 `manipulation.json` 元数据文件。我们可以直接将其打包为 ZIP 供感知实验（如 E-Prime, PsychoPy, jsPsych）使用：


In [ ]:
# 打包生成的实验刺激与元数据
import shutil

shutil.make_archive("aris_experiment_stimuli", "zip", "out/stimuli")
print("✓ 刺激已压缩打包为 aris_experiment_stimuli.zip！")

# 若在 Google Colab 中运行，取消以下注释即可直接触发浏览器下载：
# from google.colab import files
# files.download("aris_experiment_stimuli.zip")
